# extract_estreams_meteorology_from_zip notebook

This notebook embeds the full code from `extract_estreams_meteorology_from_zip.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Copy per-basin EStreams meteorology CSVs out of the official Zenodo EStreams.zip archive.

The archive layout (see Zenodo README) includes paths ending with:
  meteorology/estreams_meteorology_<basin_id>.csv

Usage (PowerShell, after you have EStreams.zip locally):
  python extract_estreams_meteorology_from_zip.py --zip_path "D:\\data\\EStreams.zip" ^
    --basins BEWA0120,BEWA0040,BEWA0019,FR000221,DENW1147
"""

from __future__ import annotations

import argparse
import shutil
import zipfile
from pathlib import Path


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Extract estreams_meteorology_*.csv files from EStreams.zip")
    p.add_argument("--zip_path", required=True, help="Path to EStreams.zip from Zenodo")
    p.add_argument(
        "--basins",
        required=True,
        help="Comma-separated basin IDs (e.g. BEWA0120,BEWA0040,...)",
    )
    p.add_argument(
        "--out_dir",
        default="data/csv_estream",
        help="Directory to write estreams_meteorology_<id>.csv (default: data/csv_estream)",
    )
    p.add_argument(
        "--list_match",
        action="store_true",
        help="Only print zip members that would match the first basin (no extraction)",
    )
    return p.parse_known_args()[0]


def find_zip_member(z: zipfile.ZipFile, basin_id: str) -> str:
    suffix = f"estreams_meteorology_{basin_id}.csv"
    norm = [n.replace("\\", "/") for n in z.namelist()]
    hits = [n for n in norm if n.endswith(suffix)]
    if not hits:
        sample = [n for n in norm if "meteorology" in n.lower() and n.endswith(".csv")][:5]
        hint = f" Sample meteorology paths: {sample}" if sample else ""
        raise FileNotFoundError(f"No zip entry ending with {suffix!r}.{hint}")
    if len(hits) > 1:
        raise RuntimeError(f"Ambiguous zip entries for {basin_id!r}: {hits}")
    # Map back to raw namelist entry (preserve original separators)
    target = hits[0]
    for raw in z.namelist():
        if raw.replace("\\", "/") == target:
            return raw
    return hits[0]


def main() -> None:
    args = parse_args()
    zip_path = Path(args.zip_path)
    if not zip_path.is_file():
        raise SystemExit(f"Zip not found: {zip_path}")

    basins = [b.strip() for b in args.basins.split(",") if b.strip()]
    if not basins:
        raise SystemExit("No basins given.")

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        if args.list_match:
            m = find_zip_member(z, basins[0])
            print(f"First basin {basins[0]!r} -> {m!r}")
            return

        for basin_id in basins:
            member = find_zip_member(z, basin_id)
            dest = out_dir / f"estreams_meteorology_{basin_id}.csv"
            with z.open(member, "r") as src, open(dest, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"Extracted {member!r} -> {dest}")


# Execute the script entry point
main()
